### Import thu vien

Cell nay nap cac thu vien can dung cho pipeline embedding va semantic search: doc JSON lon bang `ijson`, xu ly bang `numpy`/`pandas`, tao vector bang `transformers`/`torch`, va build index bang `faiss`.

In [4]:
import json
import ijson
import faiss
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from pathlib import Path

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

Config 

### Cau hinh duong dan va model

Cell nay khai bao file chunks dau vao, thu muc luu embedding/index dau ra va model E5 se dung. `OUTPUT_DIR.mkdir(...)` dam bao thu muc ket qua ton tai truoc khi ghi file.

In [5]:
CHUNKS_PATH = Path("../data/chunks/corpus_chunks_v2_e5.json")
OUTPUT_DIR = Path("../data/embeddings/e5_base")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "intfloat/multilingual-e5-base"

### Cau hinh batch va kich thuoc xu ly

Cell nay dat cac tham so chay embedding: so mau moi batch, do dai token toi da, so vector moi shard, gioi han ban ghi khi thu nghiem va cot text dung de encode.

In [12]:
BATCH_SIZE = 16 
MAX_LENGTH = 512
SHARD_SIZE = 10_000
MAX_RECORDS = 1000  
TEXT_COL = "raw_text"

Load Model

### Load tokenizer va model E5

Cell nay chon GPU neu co, dung `float16` tren CUDA de tiet kiem bo nho, sau do load tokenizer/model `multilingual-e5-base` va dua model ve che do evaluation.

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("device:", device)
print("dtype:", dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
).to(device)

model.eval()

device: cuda
dtype: torch.float16


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=Tru

Hàm encode chuẩn cho E5

### Mean pooling token embeddings

Cell nay dinh nghia cach gom embedding cua tung token thanh mot vector dai dien cho ca doan text. `attention_mask` giup bo qua padding token khi tinh trung binh.

In [21]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state.float() * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

### Ham encode danh sach van ban

Cell nay bien danh sach text thanh ma tran embedding. Moi batch se duoc tokenize, dua qua model, mean pooling, normalize L2 va tra ve `float32` de FAISS co the index/search on dinh.

In [22]:
@torch.no_grad()
def encode_texts(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            outputs = model(**inputs)

        embeddings = mean_pooling(
            outputs.last_hidden_state,
            inputs["attention_mask"],
        )

        embeddings = F.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(
            embeddings.cpu().numpy().astype("float32")
        )

        del inputs, outputs, embeddings

        if device == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(all_embeddings)

### Doc file chunks dang JSON array

Cell nay tao generator doc tung item trong file JSON lon bang `ijson`. Cach nay tranh load toan bo corpus vao RAM cung luc.

In [23]:
def iter_chunks_json_array(path):
    with open(path, "rb") as f: 
        for item in ijson.items(f, "item"):
            yield item 

### Luu embedding shard va metadata

Cell nay luu vector cua mot shard vao `.npy` va metadata tuong ung vao `.parquet`. Hai file co cung `shard_id` de sau nay ghep lai dung thu tu.

In [24]:
def save_shard(shard_id, embeddings, metadata_rows): 
    emb_path = OUTPUT_DIR / f"embeddings_part_{shard_id:05d}.npy"
    meta_path = OUTPUT_DIR / f"metadata_part_{shard_id:05d}.parquet"
    embeddings = np.asarray(embeddings, dtype="float32")
    metadata=pd.DataFrame(metadata_rows)
    
    np.save(emb_path,embeddings) 
    metadata.to_parquet(meta_path, index=False) 
    
    print("saved:", emb_path, embeddings.shape)
    print("saved:", meta_path, metadata.shape)

### Tao embedding cho corpus

Cell nay lap qua cac chunk, them prefix `passage: ` theo chuan E5, encode theo batch, gom thanh shard va luu ra disk. `MAX_RECORDS` dang dung de gioi han khi chay thu; dat `None` neu muon encode toan bo corpus.

In [25]:
batch_texts = []
batch_meta = []

shard_embeddings = []
shard_metadata = []

shard_id = 0
total_records = 0
total_embedded = 0

for item in tqdm(iter_chunks_json_array(CHUNKS_PATH)):
    if MAX_RECORDS is not None and total_records >= MAX_RECORDS:
        break
    
    total_records += 1 
    
    text = str(item.get(TEXT_COL, "")).strip()
    
    if not text:
        continue 
    
    batch_texts.append("passage: " + text)
    batch_meta.append({
        "doc_id": item.get("doc_id"),
        "chunk_id": item.get("chunk_id"),
        "url": item.get("url"),
        "topic": item.get("topic"),
        "text": text,
    })
    
    if len(batch_texts) >= BATCH_SIZE : 
        embeddings = encode_texts(batch_texts)
        shard_embeddings.append(embeddings)
        shard_metadata.extend(batch_meta)
        
        total_embedded += len(batch_texts) 
        batch_texts = []
        batch_meta = []
        
        if len(shard_metadata) >= SHARD_SIZE:
            shard_embeddings = np.vstack(shard_embeddings)

            save_shard(
                shard_id=shard_id,
                embeddings=shard_embeddings,
                metadata_rows=shard_metadata,
            )

            shard_id += 1
            shard_embeddings = []
            shard_metadata = []

if batch_texts:
    embeddings = encode_texts(batch_texts)

    shard_embeddings.append(embeddings)
    shard_metadata.extend(batch_meta)

    total_embedded += len(batch_texts)
    
if shard_metadata:
    shard_embeddings = np.vstack(shard_embeddings)

    save_shard(
        shard_id=shard_id,
        embeddings=shard_embeddings,
        metadata_rows=shard_metadata,
    )

print("total_records:", total_records)
print("total_embedded:", total_embedded)

1000it [00:12, 77.15it/s]


saved: ..\data\embeddings\e5_base\embeddings_part_00000.npy (1000, 768)
saved: ..\data\embeddings\e5_base\metadata_part_00000.parquet (1000, 5)
total_records: 1000
total_embedded: 1000


## Build FAISS Index

### Kiem tra cac shard da tao

Cell nay tim tat ca file embedding va metadata da luu trong `OUTPUT_DIR`, sau do in so shard de dam bao du lieu dau vao cho buoc build FAISS la day du.

In [26]:
embedding_files = sorted(OUTPUT_DIR.glob("embeddings_part_*.npy"))
metadata_files = sorted(OUTPUT_DIR.glob("metadata_part_*.parquet"))

print("embedding shards:", len(embedding_files))
print("metadata shards:", len(metadata_files))

embedding shards: 1
metadata shards: 1


### Build FAISS index tu embedding shards

Cell nay doc tung shard embedding, tao `IndexFlatIP`, add vector vao index va ghep metadata lai thanh mot bang duy nhat. Vi vector da normalize nen inner product tuong duong cosine similarity.

In [28]:
index = None 
all_metadata = []
offset = 0 

for emb_file, meta_file in tqdm(list(zip(embedding_files, metadata_files))):
    embeddings = np.load(emb_file).astype("float32")
    metadata = pd.read_parquet(meta_file)
    
    if index is None: 
        dim = embeddings.shape[1] 
        index = faiss.IndexFlatIP(dim)
        
    metadata.insert(
        0,
        "embedding_index",
        np.arange(offset, offset+len(metadata))
    )
    index.add(embeddings)
    all_metadata.append(metadata)

    offset += len(metadata)

metadata = pd.concat(all_metadata, ignore_index=True)
print("faiss vectors:", index.ntotal)
print("metadata rows:", len(metadata))

100%|██████████| 1/1 [00:00<00:00, 40.47it/s]

faiss vectors: 1000
metadata rows: 1000


### Khai bao duong dan artifact semantic

Cell nay dat ten file cho FAISS index, metadata tong hop va config embedding. Cac artifact nay se duoc load lai khi search ma khong can encode corpus lai.

In [29]:
INDEX_PATH = OUTPUT_DIR / "semantic_e5_base.faiss"
METADATA_PATH = OUTPUT_DIR / "chunks_metadata.parquet"
CONFIG_PATH = OUTPUT_DIR / "embedding_config.json"

### Ghi index va metadata ra disk

Cell nay luu FAISS index vao `.faiss` va metadata vao `.parquet`. Day la ket qua chinh cua buoc indexing semantic.

In [30]:
faiss.write_index(index, str(INDEX_PATH))
metadata.to_parquet(METADATA_PATH, index=False)

### Luu config embedding

Cell nay ghi lai cac thong tin quan trong cua pipeline: model, prefix document/query, max length, metric FAISS, kich thuoc vector va so vector. Config nay giup search ve sau dung dung cach encode ban dau.

In [31]:
config = {
    "model_name": MODEL_NAME,
    "text_column": TEXT_COL,
    "document_prefix": "passage: ",
    "query_prefix": "query: ",
    "max_length": MAX_LENGTH,
    "normalize_embeddings": True,
    "faiss_metric": "inner_product",
    "embedding_dim": int(index.d),
    "num_vectors": int(index.ntotal),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

config

{'model_name': 'intfloat/multilingual-e5-base',
 'text_column': 'raw_text',
 'document_prefix': 'passage: ',
 'query_prefix': 'query: ',
 'max_length': 512,
 'normalize_embeddings': True,
 'faiss_metric': 'inner_product',
 'embedding_dim': 768,
 'num_vectors': 1000}

## Search thu 

### Load index de search thu

Cell nay doc lai FAISS index va metadata vua luu. Tu day co the search truc tiep ma khong can build lai index trong RAM.

In [32]:
index = faiss.read_index(str(INDEX_PATH))
metadata = pd.read_parquet(METADATA_PATH) 

### Ham semantic search co ban

Cell nay encode query voi prefix `query: `, search top-k vector gan nhat trong FAISS, lay metadata theo index tra ve va gan them score de hien thi ket qua.

In [33]:
def semantic_search(query, top_k = 5) : 
    query_text = "query: "  + query 
    
    query_embedding = encode_texts(
        [query_text],
        batch_size=1,
        max_length=MAX_LENGTH,
    )
    scores, ids = index.search(query_embedding, top_k)

    results = []
    
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue

        row = metadata.iloc[idx].to_dict()
        row["score"] = float(score)
        results.append(row)

    return pd.DataFrame(results)

### Vi du search ve cuop tiem vang

Cell nay chay thu `semantic_search` voi mot query tieng Viet ngan. Ket qua giup kiem tra nhanh index co tra ve cac chunk lien quan ve mat ngu nghia hay khong.

In [36]:
semantic_search("cướp tiệm vàng", top_k=5)

,embedding_index,doc_id,chunk_id,url,topic,text,score
0,495,131,131_1,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,0.882593
1,494,131,131_0,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,0.874453
2,1,0,0_1,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",0.871581
3,649,176,176_0,https://kenh14.vn/bat-ngo-voi-danh-tinh-doi-tu...,Xã Hội,Bất ngờ với danh tính đối tượng dùng súng AK c...,0.870130
4,2,0,0_2,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",0.860817


### Vi du search ve quan doi va Thai Binh Duong

Cell nay thu mot query dai hon, co ngu canh quoc te. Muc dich la xem semantic search co tim duoc cac bai/chunk lien quan ngay ca khi tu khoa khong trung khop hoan toan.

In [35]:
semantic_search("quân đội Trung Quốc ở Thái Bình Dương", top_k=5)

,embedding_index,doc_id,chunk_id,url,topic,text,score
0,563,148,148_2,https://thanhnien.vn/chien-luoc-cua-my-o-indo-...,Thế giới,Chiến lược của Mỹ ở Indo-Pacific giữa sự trỗi ...,0.842808
1,561,148,148_0,https://thanhnien.vn/chien-luoc-cua-my-o-indo-...,Thế giới,Chiến lược của Mỹ ở Indo-Pacific giữa sự trỗi ...,0.838629
2,562,148,148_1,https://thanhnien.vn/chien-luoc-cua-my-o-indo-...,Thế giới,Chiến lược của Mỹ ở Indo-Pacific giữa sự trỗi ...,0.822338
3,572,153,153_1,https://tuoitre.vn/tin-the-gioi-1-8-dan-trung-...,Thế giới,Tin thế giới 1-8: Dân Trung Quốc phản đối nhà ...,0.817237
4,290,77,77_0,https://www.qdnd.vn/quoc-te/tin-tuc/my-nhat-ha...,Quốc tế,Mỹ-Nhật-Hàn tập trận phòng thủ tên lửa đạn đạo...,0.812130


## Semantic Search helpers

### Ham load artifact semantic da luu

Cell nay dong goi viec load FAISS index, metadata va config tu `OUTPUT_DIR`. Ham nay huu ich khi mo lai notebook va chi muon search, khong muon chay lai buoc tao embedding.

In [ ]:
def load_semantic_artifacts(output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)
    index_path = output_dir / "semantic_e5_base.faiss"
    metadata_path = output_dir / "chunks_metadata.parquet"
    config_path = output_dir / "embedding_config.json"

    loaded_index = faiss.read_index(str(index_path))
    loaded_metadata = pd.read_parquet(metadata_path)

    with open(config_path, "r", encoding="utf-8") as f:
        loaded_config = json.load(f)

    print("index vectors:", loaded_index.ntotal)
    print("metadata rows:", len(loaded_metadata))
    print("model:", loaded_config["model_name"])

    return loaded_index, loaded_metadata, loaded_config

### Nap artifact vao bien su dung

Cell nay goi `load_semantic_artifacts()` va gan ket qua vao `index`, `metadata`, `semantic_config` de cac ham search phia sau dung chung.

In [ ]:
index, metadata, semantic_config = load_semantic_artifacts()

### Ham semantic search nang cap

Cell nay mo rong ham search co ban: doc prefix/max length tu config, lay nhieu ung vien hon bang `candidate_k`, ho tro loc theo `topic`, loc theo `min_score`, va tra ve DataFrame da sap xep theo score.

In [ ]:
def semantic_search_v2(
    query,
    top_k=10,
    candidate_k=None,
    topic=None,
    min_score=None,
):
    if not query or not str(query).strip():
        return pd.DataFrame()

    candidate_k = candidate_k or max(top_k * 5, 50)
    candidate_k = min(candidate_k, index.ntotal)

    query_prefix = semantic_config.get("query_prefix", "query: ")
    query_text = query_prefix + str(query).strip()

    query_embedding = encode_texts(
        [query_text],
        batch_size=1,
        max_length=semantic_config.get("max_length", MAX_LENGTH),
    )

    scores, ids = index.search(query_embedding, candidate_k)
    rows = []

    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        if min_score is not None and score < min_score:
            continue

        row = metadata.iloc[int(idx)].to_dict()
        if topic is not None and row.get("topic") != topic:
            continue

        row["score"] = float(score)
        rows.append(row)

        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows)

### Dinh dang ket qua hien thi

Cell nay cat ngan cot `text` va chi giu cac cot quan trong nhu score, doc_id, chunk_id, topic, text, url. Muc dich la lam bang ket qua de doc trong notebook.

In [ ]:
def show_search_results(results, max_text_chars=220):
    if results.empty:
        return results

    display_df = results.copy()
    display_df["text"] = display_df["text"].str.slice(0, max_text_chars)

    columns = [
        "score",
        "doc_id",
        "chunk_id",
        "topic",
        "text",
        "url",
    ]
    columns = [col for col in columns if col in display_df.columns]
    return display_df[columns]

### Chay thu ham search nang cap

Cell nay goi `semantic_search_v2` voi query mau va dua ket qua qua `show_search_results()` de xem bang gon, de kiem tra nhanh chat luong search.

In [ ]:
results = semantic_search_v2("cuop tiem vang", top_k=5)
show_search_results(results)

## Gop ket qua theo bai viet

### Gop ket qua theo bai viet

Cell nay xu ly truong hop mot bai viet co nhieu chunk cung lot top. Ham giu chunk co score cao nhat cho moi `doc_id`, giup man hinh ket qua da dang hon theo bai viet.

In [ ]:
def group_results_by_doc(results, top_k=5):
    if results.empty:
        return results

    grouped = (
        results.sort_values("score", ascending=False)
        .groupby("doc_id", as_index=False)
        .agg(
            score=("score", "max"),
            chunk_id=("chunk_id", "first"),
            topic=("topic", "first"),
            text=("text", "first"),
            url=("url", "first"),
        )
        .sort_values("score", ascending=False)
        .head(top_k)
    )

    return grouped.reset_index(drop=True)

### Vi du search va gop theo bai viet

Cell nay lay nhieu chunk ung vien, gop lai theo `doc_id`, roi hien thi top bai viet. Cach nay gan voi UI search hon vi nguoi dung thuong muon xem bai viet, khong phai cac chunk trung nhau.

In [ ]:
chunk_results = semantic_search_v2("quan doi Trung Quoc Thai Binh Duong", top_k=20)
doc_results = group_results_by_doc(chunk_results, top_k=5)
show_search_results(doc_results)

Ghi chu: E5 can prefix `passage: ` khi encode document va `query: ` khi encode cau hoi. Vi embedding da normalize nen FAISS `IndexFlatIP` tuong duong cosine similarity.